# 01 – Yelp Data Exploration

Quick overview of the Yelp dataset via DuckDB.

**Pre-requisites** – Parquet files must be available at path below.

## Setup

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

PARQUET_DIR = Path("../../../Yelp-JSON/yelp_parquet")
business_files = sorted(PARQUET_DIR.glob("business/state=*/*.parquet"))
review_files = sorted(PARQUET_DIR.glob("review/year=*/*.parquet"))

con = duckdb.connect(":memory:")
print(f"DuckDB connected")
print(f"Found {len(business_files)} business Parquet files")
print(f"Found {len(review_files)} review Parquet files")

## Business Overview

In [ ]:
business_flat = str(PARQUET_DIR / "business" / "state=*" / "*.parquet")
biz_stats = con.execute(f"""
    SELECT 
        COUNT(*) as total_businesses,
        COUNT(DISTINCT state) as num_states,
        ROUND(AVG(review_count), 2) as avg_reviews,
        MIN(review_count) as min_reviews,
        MAX(review_count) as max_reviews
    FROM read_parquet('{business_flat}')
""").fetch_df()

print("Business Statistics:")
print(biz_stats)

## City Breakdown

In [ ]:
city_stats = con.execute(f"""
    SELECT 
        city, state,
        COUNT(*) as n_business,
        ROUND(AVG(stars), 2) as avg_stars,
        COUNT(DISTINCT user_id) as n_reviewers
    FROM read_parquet('{business_flat}')
    GROUP BY city, state
    ORDER BY n_business DESC
    LIMIT 15
""").fetch_df()

print(city_stats)

## Review Star Distribution

In [ ]:
review_flat = str(PARQUET_DIR / "review" / "year=*" / "*.parquet")
star_dist = con.execute(f"""
    SELECT 
        CAST(stars AS INTEGER) as stars,
        COUNT(*) as count
    FROM read_parquet('{review_flat}')
    GROUP BY CAST(stars AS INTEGER)
    ORDER BY stars
""").fetch_df()

plt.figure(figsize=(10, 5))
plt.bar(star_dist['stars'], star_dist['count'], color='steelblue')
plt.xlabel('Star Rating')
plt.ylabel('Count')
plt.title('Distribution of Review Stars')
plt.xticks([1, 2, 3, 4, 5])
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nStar distribution:")
print(star_dist)

## Reviews Over Time

In [ ]:
yearly_stats = con.execute(f"""
    SELECT 
        YEAR(date) as year,
        COUNT(*) as count
    FROM read_parquet('{review_flat}')
    GROUP BY YEAR(date)
    ORDER BY year
""").fetch_df()

plt.figure(figsize=(12, 5))
plt.plot(yearly_stats['year'], yearly_stats['count'], linewidth=2, marker='o', color='steelblue')
plt.xlabel('Year')
plt.ylabel('Number of Reviews')
plt.title('Review Count Over Time')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nReviews by year:")
print(yearly_stats)

## User-Item Sparsity (stars >= 4.0)

In [ ]:
pos = con.execute(f"""
    SELECT 
        COUNT(DISTINCT user_id) as n_u,
        COUNT(DISTINCT business_id) as n_i,
        COUNT(*) as n_int
    FROM read_parquet('{review_flat}')
    WHERE stars >= 4.0
""").fetch_df()

n_u, n_i, n_int = pos['n_u'].iloc[0], pos['n_i'].iloc[0], pos['n_int'].iloc[0]
max_possible = n_u * n_i
density = (n_int / max_possible) * 100 if max_possible > 0 else 0

print(f"Implicit feedback matrix (stars >= 4.0):")
print(f"  Users: {n_u:,}")
print(f"  Items: {n_i:,}")
print(f"  Interactions: {n_int:,}")
print(f"  Density: {density:.4f}%")

## User Activity Histogram

In [ ]:
user_counts = con.execute(f"""
    WITH user_acts AS (
        SELECT user_id, COUNT(*) as cnt
        FROM read_parquet('{review_flat}')
        WHERE stars >= 4.0
        GROUP BY user_id
    )
    SELECT cnt, COUNT(*) as num_users
    FROM user_acts
    GROUP BY cnt
    ORDER BY cnt DESC
    LIMIT 50
""").fetch_df()

plt.figure(figsize=(10, 5))
plt.loglog(user_counts['cnt'], user_counts['num_users'], 'o-', color='steelblue')
plt.xlabel('Reviews per User (log)')
plt.ylabel('Number of Users (log)')
plt.title('User Activity Distribution')
plt.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print("\nTop reviewers:")
print(user_counts.head(10))

In [ ]:
# Top business categories
categories = con.execute(f"""
    SELECT 
        categories,
        COUNT(*) as count,
        ROUND(AVG(stars), 2) as avg_stars,
        ROUND(AVG(review_count), 0) as avg_reviews
    FROM read_parquet('{business_flat}')
    WHERE categories IS NOT NULL
    GROUP BY categories
    ORDER BY count DESC
    LIMIT 20
""").fetch_df()

print("Top 20 Business Categories:")
print(categories)

In [ ]:
# Distribution of business stars
biz_stars = con.execute(f"""
    SELECT 
        ROUND(stars, 1) as rating,
        COUNT(*) as count
    FROM read_parquet('{business_flat}')
    GROUP BY ROUND(stars, 1)
    ORDER BY rating
""").fetch_df()

plt.figure(figsize=(10, 5))
plt.bar(biz_stars['rating'], biz_stars['count'], color='coral', alpha=0.8)
plt.xlabel('Business Star Rating')
plt.ylabel('Count')
plt.title('Distribution of Business Ratings')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("Business rating distribution:")
print(biz_stars)

In [ ]:
# State-level distribution
state_dist = con.execute(f"""
    SELECT 
        state,
        COUNT(*) as n_businesses,
        COUNT(DISTINCT user_id) as n_users,
        ROUND(AVG(stars), 2) as avg_rating
    FROM read_parquet('{business_flat}')
    GROUP BY state
    ORDER BY n_businesses DESC
""").fetch_df()

plt.figure(figsize=(14, 6))
plt.barh(state_dist['state'], state_dist['n_businesses'], color='steelblue')
plt.xlabel('Number of Businesses')
plt.ylabel('State')
plt.title('Ground Truth Distribution by State')
plt.tight_layout()
plt.show()

print("Top 10 states by business count:")
print(state_dist.head(10))

In [ ]:
# Top 20 most reviewed businesses
top_items = con.execute(f"""
    WITH item_count AS (
        SELECT 
            business_id,
            COUNT(*) as n_reviews,
            AVG(stars) as avg_star
        FROM read_parquet('{review_flat}')
        WHERE stars >= 4.0
        GROUP BY business_id
    )
    SELECT 
        business_id,
        n_reviews,
        ROUND(avg_star, 2) as avg_star
    FROM item_count
    ORDER BY n_reviews DESC
    LIMIT 20
""").fetch_df()

# Popularity distribution
popularity = con.execute(f"""
    WITH item_count AS (
        SELECT 
            COUNT(*) as n_reviews
        FROM read_parquet('{review_flat}')
        WHERE stars >= 4.0
        GROUP BY business_id
    )
    SELECT 
        n_reviews,
        COUNT(*) as num_items
    FROM item_count
    GROUP BY n_reviews
    ORDER BY n_reviews DESC
    LIMIT 50
""").fetch_df()

plt.figure(figsize=(12, 5))
plt.loglog(popularity['n_reviews'], popularity['num_items'], 'o-', color='green', markersize=6)
plt.xlabel('Reviews per Item (log scale)')
plt.ylabel('Number of Items (log scale)')
plt.title('Item Popularity Distribution (Power Law)')
plt.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print("Top 20 most reviewed items:")
print(top_items)
print(f"\nTotal unique items with positive feedback: {len(popularity)}")

In [ ]:
# Monthly review trends (last 24 months)
monthly_trends = con.execute(f"""
    SELECT 
        YEAR(date) as year,
        MONTH(date) as month,
        COUNT(*) as count,
        ROUND(AVG(stars), 2) as avg_rating
    FROM read_parquet('{review_flat}')
    WHERE date >= DATE '2024-01-01'
    GROUP BY YEAR(date), MONTH(date)
    ORDER BY year DESC, month DESC
    LIMIT 24
""").fetch_df()

if not monthly_trends.empty:
    plt.figure(figsize=(14, 5))
    plt.plot(range(len(monthly_trends)), monthly_trends['count'], linewidth=2, marker='o', color='purple')
    plt.xlabel('Month (Recent 24 months)')
    plt.ylabel('Number of Reviews')
    plt.title('Monthly Review Activity Trend')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("Recent monthly review trends:")
    print(monthly_trends.head(12))
else:
    print("No data available for recent months")

In [ ]:
# User engagement metrics
user_stats = con.execute(f"""
    WITH user_activity AS (
        SELECT 
            user_id,
            COUNT(*) as n_reviews,
            AVG(stars) as avg_rating,
            MIN(date) as first_review,
            MAX(date) as last_review
        FROM read_parquet('{review_flat}')
        GROUP BY user_id
    )
    SELECT 
        ROUND(AVG(n_reviews), 2) as avg_reviews_per_user,
        ROUND(STDDEV(n_reviews), 2) as stddev_reviews,
        MIN(n_reviews) as min_reviews,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY n_reviews) as median_reviews,
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY n_reviews) as p95_reviews,
        MAX(n_reviews) as max_reviews,
        COUNT(*) as total_users
    FROM user_activity
""").fetch_df()

print("User Engagement Summary:")
print(user_stats)

# Users by activity level
user_segments = con.execute(f"""
    WITH user_activity AS (
        SELECT COUNT(*) as n_reviews
        FROM read_parquet('{review_flat}')
        GROUP BY user_id
    )
    SELECT 
        'Power Users (100+ reviews)' as segment,
        COUNT(*) as count
    FROM user_activity
    WHERE n_reviews >= 100
    UNION ALL
    SELECT 
        'Active Users (10-99 reviews)',
        COUNT(*)
    FROM user_activity
    WHERE n_reviews BETWEEN 10 AND 99
    UNION ALL
    SELECT 
        'Casual Users (2-9 reviews)',
        COUNT(*)
    FROM user_activity
    WHERE n_reviews BETWEEN 2 AND 9
    UNION ALL
    SELECT 
        'One-time Users (1 review)',
        COUNT(*)
    FROM user_activity
    WHERE n_reviews = 1
""").fetch_df()

print("\nUser Segmentation:")
print(user_segments)

In [ ]:
# Review text characteristics
review_chars = con.execute(f"""
    SELECT 
        COUNT(*) as total_reviews,
        COUNT(CASE WHEN text IS NOT NULL AND LENGTH(text) > 0 THEN 1 END) as reviews_with_text,
        ROUND(AVG(CASE WHEN text IS NOT NULL THEN LENGTH(text) ELSE 0 END), 0) as avg_text_length,
        ROUND(AVG(CASE WHEN text IS NOT NULL THEN LENGTH(text) / COALESCE(NULLIF(LENGTH(REGEXP_REPLACE(text, '[^ ]', '')), 0), 1) ELSE 0 END), 1) as avg_words_per_review
    FROM read_parquet('{review_flat}')
""").fetch_df()

print("Review Text Characteristics:")
print(review_chars)

# Useful column check
useful_stats = con.execute(f"""
    SELECT 
        COUNT(*) as total_reviews,
        SUM(useful) as total_useful_votes,
        ROUND(AVG(useful), 2) as avg_useful_votes
    FROM read_parquet('{review_flat}')
""").fetch_df()

print("\nReview Usefulness (if available):")
print(useful_stats)

In [ ]:
# Recommendation dataset potential
rec_potential = con.execute(f"""
    WITH pos_feedback AS (
        SELECT 
            COUNT(DISTINCT user_id) as active_users,
            COUNT(DISTINCT business_id) as active_items,
            COUNT(*) as interactions
        FROM read_parquet('{review_flat}')
        WHERE stars >= 4.0
    ),
    all_data AS (
        SELECT 
            COUNT(DISTINCT user_id) as all_users,
            COUNT(DISTINCT business_id) as all_items,
            COUNT(*) as all_interactions
        FROM read_parquet('{review_flat}')
    )
    SELECT 
        pf.active_users,
        pf.active_items,
        pf.interactions,
        ROUND((pf.interactions::FLOAT / (pf.active_users * pf.active_items)) * 100, 4) as sparsity_pct,
        ad.all_users,
        ad.all_items,
        ad.all_interactions
    FROM pos_feedback pf, all_data ad
""").fetch_df()

print("Recommendation Dataset Potential:")
print(rec_potential)

# Coverage metrics
coverage = con.execute(f"""
    WITH positive_reviews AS (
        SELECT 
            COUNT(DISTINCT user_id) as pos_users,
            COUNT(DISTINCT business_id) as pos_items
        FROM read_parquet('{review_flat}')
        WHERE stars >= 4.0
    ),
    all_reviews AS (
        SELECT 
            COUNT(DISTINCT user_id) as all_users,
            COUNT(DISTINCT business_id) as all_items
        FROM read_parquet('{review_flat}')
    )
    SELECT 
        ROUND(100.0 * pr.pos_users / ar.all_users, 2) as active_user_coverage_pct,
        ROUND(100.0 * pr.pos_items / ar.all_items, 2) as active_item_coverage_pct
    FROM positive_reviews pr, all_reviews ar
""").fetch_df()

print("\nCoverage Metrics:")
print(coverage)

In [ ]:
print("=" * 60)
print("DATASET SUMMARY & KEY INSIGHTS")
print("=" * 60)

# Compile key metrics
summary_query = con.execute(f"""
    SELECT 
        COUNT(DISTINCT CASE WHEN stars >= 4.0 THEN user_id END) as target_users,
        COUNT(DISTINCT CASE WHEN stars >= 4.0 THEN business_id END) as target_items,
        COUNT(CASE WHEN stars >= 4.0 THEN 1 END) as target_interactions,
        COUNT(DISTINCT YEAR(date)) as year_span,
        ROUND(AVG(CASE WHEN stars >= 4.0 THEN stars END), 2) as avg_positive_rating
    FROM read_parquet('{review_flat}')
""").fetch_df()

row = summary_query.iloc[0]
print(f"\n📊 COLLABORATIVE FILTERING DATASET (stars ≥ 4.0):")
print(f"   • Users: {int(row['target_users']):,}")
print(f"   • Items: {int(row['target_items']):,}")
print(f"   • Interactions: {int(row['target_interactions']):,}")
print(f"   • Matrix density: {(int(row['target_interactions']) / (int(row['target_users']) * int(row['target_items']))) * 100:.4f}%")
print(f"   • Avg positive rating: {row['avg_positive_rating']} stars")
print(f"   • Time span: {int(row['year_span'])} years")

print(f"\n📈 DATASET CHARACTERISTICS:")
print(f"   • Highly sparse matrix (typical for recommendations)")
print(f"   • Power-law distributed user & item interactions")
print(f"   • Sufficient scale for learning latent representations")
print(f"   • Ready for ELSA pre-training + TopK SAE interpretation")

print(f"\n✅ RECOMMENDATION MODEL VIABILITY:")
print(f"   • Implicit feedback sufficient for collaborative filtering")
print(f"   • User-based and content-based hybrid possible")
print(f"   • Cold-start mitigation: substantial category/geographic data")
print("=" * 60)

## Summary & Key Insights

## Recommendation Potential Analysis

## Review Characteristics

## User Engagement Analysis

## Temporal Patterns - Monthly Trends

## Popular Items (Businesses with Most Reviews)

## Geographic Distribution

## Business Rating Distribution

## Business Category Analysis